# 🎵 EAS 587 – Spotify Analysis | Gold Layer
**Team TriForce** — Dilip, Harsha, Pavan

Reads Silver, produces two business-ready managed Gold Delta tables:
- `genre_popularity_summary` — avg audio features per genre & tier
- `tier_feature_summary` — feature averages per popularity tier

In [0]:
from pyspark.sql import functions as F

# ── Configuration ──────────────────────────────────────────────────
SILVER_TABLE = "workspace.spotify_silver.clean_tracks"
GOLD_CAT     = "workspace"
GOLD_SCH     = "spotify_gold"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CAT}.{GOLD_SCH}")
print(f"✅ Schema '{GOLD_CAT}.{GOLD_SCH}' ready")

✅ Schema 'workspace.spotify_gold' ready


In [0]:
df_silver = spark.table(SILVER_TABLE)
print(f"📥 Silver rows : {df_silver.count():,}")

📥 Silver rows : 80,140


In [0]:
# ── Gold Table 1 — Genre × Popularity Tier summary ─────────────────
df_genre_summary = (
    df_silver
    .groupBy("genre", "popularity_tier")
    .agg(
        F.count("*")                        .alias("track_count"),
        F.round(F.avg("popularity"),   2)   .alias("avg_popularity"),
        F.round(F.avg("danceability"), 3)   .alias("avg_danceability"),
        F.round(F.avg("energy"),       3)   .alias("avg_energy"),
        F.round(F.avg("valence"),      3)   .alias("avg_valence"),
        F.round(F.avg("acousticness"), 3)   .alias("avg_acousticness"),
        F.round(F.avg("tempo"),        2)   .alias("avg_tempo"),
        F.round(F.avg("loudness"),     2)   .alias("avg_loudness"),
    )
    .orderBy("genre", "popularity_tier")
)

(
    df_genre_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_CAT}.{GOLD_SCH}.genre_popularity_summary")
)

print("✅ Gold table 'genre_popularity_summary' written")

✅ Gold table 'genre_popularity_summary' written


In [0]:
# ── Gold Table 2 — Feature averages per Popularity Tier ────────────
df_tier_features = (
    df_silver
    .groupBy("popularity_tier")
    .agg(
        F.count("*")                              .alias("track_count"),
        F.round(F.avg("popularity"),         2)   .alias("avg_popularity"),
        F.round(F.avg("danceability"),       3)   .alias("avg_danceability"),
        F.round(F.avg("energy"),             3)   .alias("avg_energy"),
        F.round(F.avg("valence"),            3)   .alias("avg_valence"),
        F.round(F.avg("acousticness"),       3)   .alias("avg_acousticness"),
        F.round(F.avg("instrumentalness"),   3)   .alias("avg_instrumentalness"),
        F.round(F.avg("tempo"),              2)   .alias("avg_tempo"),
        F.round(F.avg("loudness"),           2)   .alias("avg_loudness"),
        F.round(F.avg("speechiness"),        3)   .alias("avg_speechiness"),
    )
    .orderBy("popularity_tier")
)

(
    df_tier_features.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_CAT}.{GOLD_SCH}.tier_feature_summary")
)

print("✅ Gold table 'tier_feature_summary' written")

✅ Gold table 'tier_feature_summary' written


## ✅ Validation — Gold tables exist

In [0]:
%sql
SHOW TABLES IN workspace.spotify_gold;

database,tableName,isTemporary
spotify_gold,genre_popularity_summary,false
spotify_gold,tier_feature_summary,false


In [0]:
%sql
-- ⭐ DEMO QUERY: What makes a High-popularity track?
SELECT
    popularity_tier,
    track_count,
    avg_popularity,
    avg_danceability,
    avg_energy,
    avg_valence,
    avg_tempo,
    avg_loudness
FROM workspace.spotify_gold.tier_feature_summary
ORDER BY avg_popularity DESC;

popularity_tier,track_count,avg_popularity,avg_danceability,avg_energy,avg_valence,avg_tempo,avg_loudness
High,9800,67.53,0.595,0.638,0.485,120.51,-7.75
Medium,39323,44.08,0.572,0.634,0.469,123.03,-8.28
Low,31017,18.64,0.544,0.653,0.463,123.0,-8.83


In [0]:
%sql
-- Top genres by High-popularity track count
SELECT genre, track_count, avg_popularity, avg_danceability, avg_energy
FROM workspace.spotify_gold.genre_popularity_summary
WHERE popularity_tier = 'High'
ORDER BY track_count DESC
LIMIT 15;

genre,track_count,avg_popularity,avg_danceability,avg_energy
k-pop,482,69.09,0.661,0.729
pop-film,430,64.45,0.595,0.604
hip-hop,384,67.86,0.708,0.687
chill,324,64.84,0.68,0.434
alt-rock,302,70.74,0.525,0.732
edm,295,68.82,0.65,0.744
dance,276,77.33,0.681,0.682
electro,261,69.93,0.659,0.578
hard-rock,259,68.51,0.456,0.78
grunge,259,64.94,0.454,0.801


In [0]:
%sql
-- Delta versioning — great to show during demo!
DESCRIBE HISTORY workspace.spotify_gold.tier_feature_summary;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-04-17T00:17:33.000Z,71847892219537,linkeddg9669@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(479321165056635),61ac3668-9785-402a-a7df-3efac53fbd90,0416-232715-9tjm0lqp-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 3127)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
